In [1]:
from fom import create_fom
import numpy as np

In [ ]:
# parameterstudy = {}   # here we collect all the information (customized initial condition, final time)

# # run this cell once and then the one below for all parameters, then compute the snapshots

In [12]:
import numpy as np
from scipy.signal import find_peaks


# parameters we want to compute (and test against)
# mus = [2.75, 3.25, 3.75, 4.25] ([3.0,3.5])

mu = 3.5

# 1. let FOM run for a longer time to reach the limit cycle
t_long = 30
fom_long = create_fom(nt=700, h=50, time=t_long) 
u_long = fom_long.solve(mu)
    
# 2. compute L2-norm of the long solution
m_op = fom_long.mass
norms = u_long.norm(m_op)

# create times array that exactly matches the trajectory length
times = np.linspace(0, t_long, len(u_long)) 
    
# 3. find all peaks in the norms
peak_indices, _ = find_peaks(norms)
peak_values = norms[peak_indices]
peak_times = times[peak_indices]

# 4. compute exact period from the highest peaks 
top_indices = np.argsort(peak_values)[-2:]       
top_peak_times = peak_times[top_indices]
sorted_top_times = np.sort(top_peak_times)

T_per = sorted_top_times[-1] - sorted_top_times[-2]
    
# 5. take exact state at the last (highest) peak as initial cond
actual_top_indices = np.sort(peak_indices[top_indices])
last_peak_index = actual_top_indices[-1]

periodic_u0 = u_long[last_peak_index].to_numpy() 
    
parameterstudy[mu] = {
    'period': T_per,
    'initial_data': periodic_u0
}
    
print("Currently safed in parameterstudy:")
for m, data in parameterstudy.items():
    print(f"{m} -> T = {data['period']:.4f}")

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

Currently safed in parameterstudy:
2.75 -> T = 6.7286
3.25 -> T = 7.3714
3.75 -> T = 8.3143
4.25 -> T = 9.4714
3.0 -> T = 7.0286
3.5 -> T = 7.8429


In [13]:
# safe parameter_study to access in another notebook

from pymor.core.pickle import dump

with open('brusselator_parameters.pkl', 'wb') as f:
    dump(parameterstudy, f)

In [14]:
# access parameter study

from pymor.core.pickle import load

with open('brusselator_parameters.pkl', 'rb') as f:
    parameterstudy = load(f)

print(f"Mus we got: {list(parameterstudy.keys())}")
print("Currently safed in parameterstudy:")
for m, data in parameterstudy.items():
    print(f"{m} -> T = {data['period']:.4f}")

Mus we got: [2.75, 3.25, 3.75, 4.25, 3.0, 3.5]
Currently safed in parameterstudy:
2.75 -> T = 6.7286
3.25 -> T = 7.3714
3.75 -> T = 8.3143
4.25 -> T = 9.4714
3.0 -> T = 7.0286
3.5 -> T = 7.8429
